<a href="https://colab.research.google.com/github/StereoWings7/potential-octo-giggle/blob/main/transfer_example.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import sys
import time
import copy

import torch
import torchvision
from torchvision import models
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder

os.chdir('/content/drive/MyDrive/Colab Notebooks')

In [ ]:
#データ絡みのパラメータを指定しておく
batch_size = 32
num_epochs = 20

In [ ]:
#学習用のデータセットとしてCIFAR100をダウンロードする
#Normalize用のmean,stdはCIFAR-100の値を用いる
#https://gist.github.com/weiaicunzai/e623931921efefd4c331622c344d8151
data_transforms = {
    'train': transforms.Compose([
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize([0.5071, 0.4867, 0.4408], [0.2675, 0.2565, 0.2761])
    ]),
    'val': transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize([0.5071, 0.4867, 0.4408], [0.2675, 0.2565, 0.2761])
    ]),
}

trainset = torchvision.datasets.CIFAR100(root='./data', train=True,download=True, transform=data_transforms['train'])
trainloader = torch.utils.data.DataLoader(trainset, batch_size=batch_size,shuffle=True)

testset = torchvision.datasets.CIFAR100(root='./data', train=False,download=True, transform=data_transforms['val'])
testloader = torch.utils.data.DataLoader(testset, batch_size=batch_size,shuffle=False)

dataloaders = {'train': trainloader, 'val': testloader}
dataset_sizes = {'train': len(trainset), 'val': len(testset)}

Files already downloaded and verified
Files already downloaded and verified


In [ ]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(device)

cuda:0


In [ ]:
#ImageNetで1000クラス分類pretrainedされた重み付け係数をロードする
model_ft = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
100%|██████████| 44.7M/44.7M [00:00<00:00, 54.2MB/s]


In [ ]:
#モデルの構造を見る。ImageNet用に、最後のLinear層が1000クラス出力になっていることに注意
for x in list(model_ft.children()):
  print(x, '\n')

Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False) 

BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True) 

ReLU(inplace=True) 

MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False) 

Sequential(
  (0): BasicBlock(
    (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  )
  (1): BasicBlock(
    (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), 

In [ ]:
#CIFAR-100用として、最後の線形層を100クラス分類に書き換える
num_ftrs = model_ft.fc.in_features
model_ft.fc = torch.nn.Linear(num_ftrs, 100)

In [ ]:
#線形層が書き変わったか確認
for x in list(model_ft.children()):
  print(x, '\n')

Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False) 

BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True) 

ReLU(inplace=True) 

MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False) 

Sequential(
  (0): BasicBlock(
    (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  )
  (1): BasicBlock(
    (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), 

上記の例では(というか一般にtorchvision.modelsからロードすると)NNの最後にsoftmaxが乗ってなくて、線形層で終わっている(こんな感じで線形層の乗せ替えとかをやりやすくするためと思われる)。したがって、modelの出力からクラス分類の確率を求めるにはsoftmaxをかける必要がある。(なおtorch.nn.CrossEntropyLoss()はlog(softmax(x))みたいな定義になってるので、↑のような「いちばん上の層が線形層」みたいなモデルの出力をそのまま放り込んで大丈夫、なはず)

In [ ]:
#モデルについて、headの乗せ替えができたらGPUに移動させる
model_ft = model_ft.to(device)

In [ ]:
#学習用ループを定義する
def train_model(model, criterion, optimizer, scheduler, num_epochs):
    since = time.time()

    best_model_wts = copy.deepcopy(model.state_dict())
    best_acc = 0.0

    for epoch in range(num_epochs):
        print('Epoch {}/{}'.format(epoch, num_epochs - 1))
        print('-' * 10)

        # Each epoch has a training and validation phase
        for phase in ['train', 'val']:
            if phase == 'train':
                scheduler.step()
                model.train()  # Set model to training mode
            else:
                model.eval()   # Set model to evaluate mode

            running_loss = 0.0
            running_corrects = 0

            # Iterate over data.
            for inputs, labels in dataloaders[phase]:
                inputs = inputs.to(device)
                labels = labels.to(device)

                # zero the parameter gradients
                optimizer.zero_grad()

                # forward
                # track history if only in train
                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    #torch.maxの第2引数は、どの軸に向けて最大値をとるか指定する。
                    #第1軸方向は100クラス分類の確率
                    #torch.maxの戻り値をunpackして、2つめはmaxとなったindicesが返ってくる
                    _, preds = torch.max(torch.nn.Softmax(dim=1)(outputs), 1)
                    #lossにはCrossEntropyLossを利用する予定なので、モデル出力をそのまま利用する。
                    loss = criterion(outputs, labels)

                    # backward + optimize only if in training phase
                    if phase == 'train':
                        loss.backward()
                        optimizer.step()

                # statistics
                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)

            epoch_loss = running_loss / dataset_sizes[phase]
            epoch_acc = running_corrects.double() / dataset_sizes[phase]

            print('{} Loss: {:.4f} Acc: {:.4f}'.format(
                phase, epoch_loss, epoch_acc))

            # deep copy the model
            if phase == 'val' and epoch_acc > best_acc:
                best_acc = epoch_acc
                best_model_wts = copy.deepcopy(model.state_dict())

        print()

    time_elapsed = time.time() - since
    print('Training complete in {:.0f}m {:.0f}s'.format(
        time_elapsed // 60, time_elapsed % 60))
    print('Best val Acc: {:4f}'.format(best_acc))

    # load best model weights
    model.load_state_dict(best_model_wts)
    return model

In [ ]:
#学習用パラメータを指定
criterion = torch.nn.CrossEntropyLoss()
optimizer= torch.optim.SGD(model_ft.parameters(), lr=0.001, momentum=0.9)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.9)


In [ ]:
# training
model_ft = train_model(model_ft, criterion, optimizer, scheduler,num_epochs)

Epoch 0/19
----------
train Loss: 2.7222 Acc: 0.3201
val Loss: 2.0635 Acc: 0.4475

Epoch 1/19
----------
train Loss: 1.9812 Acc: 0.4649
val Loss: 1.8016 Acc: 0.5054

Epoch 2/19
----------
train Loss: 1.7058 Acc: 0.5288
val Loss: 1.7221 Acc: 0.5298

Epoch 3/19
----------
train Loss: 1.4896 Acc: 0.5839
val Loss: 1.6505 Acc: 0.5504

Epoch 4/19
----------
train Loss: 1.3517 Acc: 0.6143
val Loss: 1.6428 Acc: 0.5587

Epoch 5/19
----------
train Loss: 1.2420 Acc: 0.6409
val Loss: 1.5927 Acc: 0.5750

Epoch 6/19
----------
train Loss: 1.1300 Acc: 0.6709
val Loss: 1.6106 Acc: 0.5753

Epoch 7/19
----------
train Loss: 1.0461 Acc: 0.6922
val Loss: 1.5999 Acc: 0.5759

Epoch 8/19
----------
train Loss: 0.9406 Acc: 0.7205
val Loss: 1.6118 Acc: 0.5811

Epoch 9/19
----------
train Loss: 0.8618 Acc: 0.7412
val Loss: 1.6146 Acc: 0.5838

Epoch 10/19
----------
train Loss: 0.8015 Acc: 0.7573
val Loss: 1.6579 Acc: 0.5876

Epoch 11/19
----------
train Loss: 0.7400 Acc: 0.7761
val Loss: 1.6401 Acc: 0.5915

Ep